# FIFA World Cup 2026 — Scoreboard Prediction

### Goal

Predict the final scoreline (home goals vs. away goals) for matches at the
2026 FIFA World Cup, using historical international results and team/player
strength ratings as inputs.

### Data

- **`df_results`** — international match results, 1872–present
  (`datasets/international_results-master/results.csv`), the historical
  record of goals scored by each side in each match.
- **`df_formernames`** — historical name changes for national teams
  (`datasets/international_results-master/former_names.csv`), used to
  reconcile teams that have been renamed over time so history isn't split
  across two identities.
- **`df_elo`** — Elo ratings for World Cup 2026 squads
  (`datasets/elo-ratings/elo_ratings_wc2026.csv`), a single strength number
  per team going into the tournament.
- **`df_eafc`** — EA FC 26 player ratings
  (`datasets/FC26_20250921.csv`), individual player attributes that can be
  aggregated into a squad-strength feature.

### Approach

1. Clean and merge the datasets above into one match-level table, with each
   row describing a historical match and the pre-match strength of both sides.
2. Engineer features available *before kickoff* (team Elo/rating, recent
   form, head-to-head history) — never anything that leaks the result.
3. Model the two scores (home goals, away goals) — e.g. independent Poisson
   regressions per side — and evaluate against held-out historical matches
   using a time-based split.
4. Apply the trained model to the 2026 World Cup fixtures to predict a
   scoreboard for each match.

In [22]:
import pandas as pd
import numpy as np
import sklearn as skl

## Datasets analysis

In [23]:
df_elo = pd.read_csv('datasets/elo-ratings/elo_ratings_wc2026.csv')

In [24]:
df_eafc = pd.read_csv('datasets/FC26_20250921.csv')
df_eafc.drop(columns=['player_url', 'fifa_version', 'fifa_update', 'long_name', 'player_face_url'])

/tmp/ipykernel_5444/1608615350.py:1: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eafc = pd.read_csv('datasets/FC26_20250921.csv')


,player_id,fifa_update_date,short_name,player_positions,overall,potential,value_eur,wage_eur,age,dob,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,252371,2025-09-19,J. Bellingham,"CAM, CM",90,94,174500000,320000,22,2003-06-29,...,85+3,85+3,85+3,83+3,82+3,81+3,81+3,81+3,82+3,18+3
1,239053,2025-09-19,F. Valverde,"CM, CDM, RB",89,90,120500000,340000,26,1998-07-22,...,87+3,87+3,87+3,86+3,86+3,83+3,83+3,83+3,86+3,18+3
2,212622,2025-09-19,J. Kimmich,"CDM, RB, CM",89,89,86000000,140000,30,1995-02-08,...,87+2,87+2,87+2,86+3,85+3,82+3,82+3,82+3,85+3,21+3
3,235212,2025-09-19,A. Hakimi,"RB, RM",89,90,111000000,170000,26,1998-11-04,...,83+3,83+3,83+3,86+3,86+3,81+3,81+3,81+3,86+3,17+3
4,224232,2025-09-19,N. Barella,CM,87,87,79500000,69000,28,1997-02-07,...,85+2,85+2,85+2,84+3,83+3,80+3,80+3,80+3,83+3,19+3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18400,267946,2025-09-19,Lim Jun Sub,GK,48,54,60000,500,21,2003-08-22,...,19+2,19+2,19+2,16+2,16+2,19+2,19+2,19+2,16+2,47+2
18401,76593,2025-09-19,Yu Jianxian,GK,49,56,80000,500,23,2001-11-06,...,18+2,18+2,18+2,16+2,16+2,19+2,19+2,19+2,16+2,48+2
18402,77205,2025-09-19,Park Man Ho,GK,48,59,90000,500,21,2004-02-28,...,19+2,19+2,19+2,17+2,16+2,17+2,17+2,17+2,16+2,47+2
18403,278149,2025-09-19,D. Chauhan,GK,51,61,110000,500,21,2003-11-20,...,17+2,17+2,17+2,17+2,17+2,16+2,16+2,16+2,17+2,50+2


In [25]:
df_formernames = pd.read_csv('datasets/international_results-master/former_names.csv')

In [26]:
df_results = pd.read_csv('datasets/international_results-master/results.csv')